# Langchain Agents 
This notebook demonstrates how to create and use agents in Langchain, including the use of tools like DuckDuckGoSearch for real-time information retrieval.

At the end of this notebook, we can see how to create a ReAct agent from a prompt template and use it with a memory system to maintain chat history.



In [1]:
from dotenv import load_dotenv
from langchain_groq import ChatGroq


In [2]:
# load environment variables from .env file
load_dotenv()

True

In [3]:
# Get llms models via the `langchain_groq` package
llm_model = ChatGroq(
    model="llama-3.3-70b-versatile",            #llama-3.3-70b-versatile",
    temperature=.5,
    max_tokens=512,
)

## Langchain Tools

The [third-party tools](https://python.langchain.com/docs/integrations/tools/) from the langchain-community ecosytem allow us to extent the capabilities of our agents

In [4]:
# from langchain_community.tools import DuckDuckGoSearchRun
from ddgs import DDGS
from langchain_community.utilities import DuckDuckGoSearchAPIWrapper

`DuckDuckGoSearchRun` allows us to perform web searches and retrieve information in real-time, which can be very useful for agents that need to answer questions based on current data.


In [5]:
#api_wrapper = DuckDuckGoSearchAPIWrapper(region="en-en",max_results=10)
#search = DuckDuckGoSearchRun()#,api_wrapper=api_wrapper)

def ddg_search(query: str):
    with DDGS() as ddgs:
        return ddgs.text(query, max_results=5)
search = ddg_search

Why did they comment out api_wrapper and search?

Because they originally intended to use LangChain’s built-in DuckDuckGoSearchRun, but then switched to using their own ddg_search function instead.

They don’t use them.

In [6]:
# This is just a sanity check: “Does the DDG wrapper actually return results?”
results = search("Prime Italian Minister")  

In [7]:
results

[{'title': 'Prime Minister of Italy - Wikipedia',
  'href': 'https://en.wikipedia.org/wiki/Prime_Minister_of_Italy',
  'body': 'The Italian prime minister presided over a very unstable political system as in its first sixty years of existence (1861–1921) Italy changed its ...'},
 {'title': 'List of prime ministers of Italy - Wikipedia',
  'href': 'https://en.wikipedia.org/wiki/List_of_Prime_Ministers_of_Italy',
  'body': 'Top right: Benito Mussolini was the longest-serving prime minister in Italian history. ... was the longest-serving prime minister of the Italian ...'},
 {'title': 'Former Italian prime minister Silvio Berlusconi dies aged 86 -',
  'href': 'https://www.dailymotion.com/video/x8lp8iu',
  'body': 'Former Italian prime minister Silvio Berlusconi dies aged 86 ... Silvio Berlusconi: Former Italian prime minister dies aged 86'},
 {'title': "In White House meeting, Italian prime minister vows to 'make",
  'href': 'https://www.upi.com/Top_News/US/2025/04/17/italys-meloni-voes-t

In [8]:
# Define a tool for the DuckDuckGo search
from langchain.tools import Tool
search_tool = Tool(
    name="DuckDuckGoSearch",
    func=ddg_search,
    description="Use this tool when you need to search for real-time information from DuckDuckGo."
)

In [9]:
from langchain.agents import initialize_agent, AgentType


In [10]:
# Initialize the agent with the DuckDuckGo search tool
agent = initialize_agent(
    tools=[search_tool],
    llm=llm_model,
    agent_type=AgentType.ZERO_SHOT_REACT_DESCRIPTION,
    verbose=True,
    max_iterations=7,
    agent_name="DuckDuckGoSearchAgent",
    agent_description="An agent that can search the web using DuckDuckGoSearch and summarize findings clearly."
)

# Now we can use the agent to answer questions
results = agent.invoke("Who is the Italian Prime Minister?")  # Example query to the agent

/Users/alastair/Github/ds-ai-agent/.venv/lib/python3.11/site-packages/langchain_core/_api/deprecation.py:119: LangChainDeprecationWarning: The function `initialize_agent` was deprecated in LangChain 0.1.0 and will be removed in 0.3.0. Use Use new agent constructor methods like create_react_agent, create_json_agent, create_structured_chat_agent, etc. instead.
  warn_deprecated(




> Entering new AgentExecutor chain...
To find out who the current Italian Prime Minister is, I need to search for the most recent information available.

Action: DuckDuckGoSearch
Action Input: Italian Prime Minister
Observation: [{'title': 'Italian Prime Minister', 'href': 'https://en.wikipedia.org/wiki/Italian_Prime_Minister', 'body': "The prime minister of Italy, officially the president of the Council of Ministers (Italian: Presidente del Consiglio dei ministri), is the head of government of the Italian Republic. The office of president of the Council of Ministers is established by articles 92–96 of the Constitution of Italy; the president of the Council of Ministers is appointed by the president of the Republic and must have the confidence of the Parliament to stay in office.Prior to the establishment of the Italian Republic, the position was called the President of the Council of Ministers of the Kingdom of Italy (Presidente del Consiglio dei ministri del Regno d'Italia). From 1

ValueError: An output parsing error occurred. In order to pass this error back to the agent and have it try again, pass `handle_parsing_errors=True` to the AgentExecutor. This is the error: Could not parse LLM output: `Based on the search results, I have found that the current Italian Prime Minister is Giorgia Meloni, who has been in office since October 22, 2022, and is the first woman to hold the position in the history of the Italian Republic.

Action: None`

### Create a an Agent with DuckDuckGoSearch with Memory
In this section, we will create an agent that can search the web using DuckDuckGoSearch and maintain a memory of previous interactions. 

In [11]:
from langchain.prompts import PromptTemplate

In [12]:
react_template = """
'Answer the following questions as best you can. You have access to the following tools:

{tools}
Use the following format:

Question: the input question you must answer
Thought: you should always think about what to do
Action: the action to take, should be one of [{tool_names}]
Action Input: the input to the action
Observation: the result of the action
... (this Thought/Action/Action Input/Observation can repeat N times)
Thought: I now know the final answer
Final Answer: the final answer to the original input question

Begin!
Question: {input}
Thought:{agent_scratchpad}'
"""

The react_template instructs the language model to:
1. __Think__ about how to solve a user's question
2. __Use tools__ (actions) to gather more information or perform actions
3. __Observe results__ and update the its thoughts
4. Eventually, reach a __Final Answer__

The following is the explanation of the `react_template` used to create this agent:

```python
react_template = """
'Answer the following questions as best you can. You have access to the following tools:

{tools}
```

This introduces the agent and tells it what tools it can use
+ {tools} is a placeholder for the tools that will be used by the agent

```python
"""
Question: the input question you must answer
Thought: you should always think about what to do
Action: the action to take, should be one of [{tool_names}]
Action Input: the input to the action
Observation: the result of the action
... (this Thought/Action/Action Input/Observation can repeat N times)
Thought: I now know the final answer
Final Answer: the final answer to the original input question
"""
```

This is the strict __format__ the LLM must follow to:

* Show its reasoning (Thought)

* Choose and execute a tool (Action)

* Feed it input (Action Input)

* Observe results (Observation)

* Loop until it gets to the final answer

```python
"""
Question: {input}
Thought:{agent_scratchpad}'
"""
```
+ `{input}` is the actual user question.

+ `{agent_scratchpad}` holds any intermediate steps already completed (e.g., previous Thoughts, Actions, Observations).

+ The agent continues from this point.

In [13]:
# Create a prompt template for the agent
prompt_template = PromptTemplate(input_variables=['agent_scratchpad', 'input', 'tool_names', 'tools'],
    template=react_template)

In [14]:
from langchain.agents import AgentExecutor, create_react_agent
from langchain.memory import ChatMessageHistory
from langchain_core.runnables.history import RunnableWithMessageHistory

In [15]:
# Create a memory for the agent
memory = ChatMessageHistory(session_id="test-session")

Noting that I have no fucking clue what this is. I don't know:
- what ChatMessageHistory does.
- what kind of output ChatMessageHistory returns.
- why you need to set a value for session_id.
- why that value is "test-session". 
- Where the value for "test-session" comes from. 

I tried asking ChatGPT, and after 6 different and specific prompts about what I wanted to know, the best it got was fucking me off with the stupid response "It creates an empty list of messages and gives it a label called "test-session". That’s all."

Fucking ridiculous and unhelpful, I know. That's what I told it. 

In [16]:
# Build a ReAct-style agent
agent = create_react_agent(llm_model, [search_tool], prompt_template)           #this is the newer version of initialize_agent()

# Wrap the agent in an executor so it can actually run:
agent_executor = AgentExecutor(
    agent=agent,                        # the agent we just created
    tools=[search_tool],                # the tools the executor is allowed to call
    verbose=True,                       # show Thought/Action/Observation output
    handle_parsing_errors=True,         # avoid crashing if the LLM formats something incorrectly
)


Until now, every agent you built used **`initialize_agent()`**, which is the older all-in-one helper. It created the agent *and* made it runnable in a single step.

This section switches to the **newer, two-step LangChain pattern**:

1. **`create_react_agent()`**
   Builds the agent’s reasoning logic (the ReAct prompt + how it chooses tools).

2. **`AgentExecutor()`**
   Is LangChain’s controller that runs the ReAct loop. It takes the agent’s Thought/Action steps, executes whatever tool the agent chooses, feeds the tool result back in, and stops when the agent produces a Final Answer.


In [17]:
# Add memory support: this wrapper lets the agent read + store past messages during a conversation
agent_with_chat_history = RunnableWithMessageHistory(
    agent_executor,                                 # the executable agent we created
    lambda session_id: memory,                      # a function that receives LangChain's session_id and returns the ChatMessageHistory object to use for this run
    input_messages_key="input",                     # which key in .invoke() contains the new user message
    history_messages_key="chat_history",            # where past messages should be inserted
)


In [18]:
# Now we can use the agent with chat history to answer questions
agent_with_chat_history.invoke(
    {"input": "How many people live in italy?"},
    config={"configurable": {"session_id": "1"}},
)

Parent run aba4d3a6-c331-4b5e-804c-07fb8fdcd386 not found for run b3ee5bf2-6080-4206-bf0b-feb0dc1d3591. Treating as a root run.




> Entering new AgentExecutor chain...
Thought: To find the current population of Italy, I should search for the most recent data available. 

Action: DuckDuckGoSearch
Action Input: population of Italy[{'title': 'Population of Italy', 'href': 'https://en.wikipedia.org/wiki/Population_of_Italy', 'body': "The demography of Italy is monitored by the Italian National Institute of Statistics (Istat).At the beginning of 2024, Italy had an estimated population of 58.9 million. Its population density, at 195.7 inhabitants per square kilometre (507/sq mi), is higher than both the EU (106.6/km2) and European (72.9/km2) average. However, the distribution of the population is very uneven: the most densely populated areas are the Po Valley (with about a third of the country's population) in northern Italy and the metropolitan areas of Rome and Naples in central and southern Italy; landlocked, rural and mountainous areas are very sparsely populated, notably the Alps and Apennines ranges, the platea

{'input': 'How many people live in italy?',
 'chat_history': [],
 'output': 'The current population of Italy is approximately 59 million inhabitants, with the most recent estimate being 59,033,170 as of January 26, 2026, according to Worldometer.'}

In [19]:
# Let's ask another question to see how the agent uses the chat history
agent_with_chat_history.invoke(
    {"input": "what is their national anthem called?"},
    config={"configurable": {"session_id": "1"}},
)

Parent run 67fe382c-9934-400a-bff0-d755919c1064 not found for run af65914f-a3ef-414d-a6c3-c49c955a6b12. Treating as a root run.




> Entering new AgentExecutor chain...
Thought: To answer this question, I need to know which country's national anthem is being referred to. However, since the country is not specified, I'll need to make an assumption or gather more information. Let's assume the user is asking about a specific country, but I don't know which one. I'll need to search for a list of national anthems or try to find a clue about the country in question.

Action: DuckDuckGoSearch
Action Input: "national anthems of the world"[{'title': 'National anthems of the world', 'href': 'https://en.wikipedia.org/wiki/National_anthems_of_the_world', 'body': 'Most nation states have an anthem, defined as "a song, as of praise, devotion, or patriotism"; most anthems are either marches or hymns in style. A song or hymn can become a national anthem under the state\'s constitution, by a law enacted by its legislature, or simply by tradition. A royal anthem is a patriotic song similar to a national anthem, but it specificall

{'input': 'what is their national anthem called?',
 'chat_history': [HumanMessage(content='How many people live in italy?'),
  AIMessage(content='The current population of Italy is approximately 59 million inhabitants, with the most recent estimate being 59,033,170 as of January 26, 2026, according to Worldometer.')],
 'output': "I'm sorry, but I couldn't find the name of the national anthem you're looking for. Could you please provide more context or specify the country you're interested in? I'll do my best to provide the name of the national anthem."}

**How memory works in LangChain**

Memory is a running list of past messages (user messages + AI messages).

Every time you call the agent:
1. LangChain **loads all previous messages** for that session.
2. It **adds them to the prompt** before the new user input.
3. The model sees the entire accumulated history.
4. The agent answers.
5. LangChain **saves the new turn** (your input + the agent’s output) into the same history list.

Any call using the same `session_id` automatically becomes part of the same conversation.


### Exercise
Create or Select some tools from the [langchain-community tools](https://python.langchain.com/docs/integrations/tools/) and create a ReAct agent that can use them to solve certain tasks.